In [0]:
# Configuration — edit table_configs to load a subset
source_volume = "/Volumes/adventure_works/default/rawdata"
ddl_file = f"{source_volume}/instawdb.sql"  # original AdventureWorks SQL Server DDL
target_schema = "adventure_works.bronze"

# CSV file -> (DDL schema, DDL table, bronze table name)
table_configs = {
    "Product.csv":             ("Production", "Product",            "dim_product"),
    "PurchaseOrderDetail.csv": ("Purchasing", "PurchaseOrderDetail", "dim_purchaseorderdetail"),
    "PurchaseOrderHeader.csv": ("Purchasing", "PurchaseOrderHeader", "dim_purchaseorderheader"),
    "SalesOrderDetail.csv":    ("Sales",      "SalesOrderDetail",   "fact_sales"),
    "SalesTerritory.csv":      ("Sales",      "SalesTerritory",     "dim_sales_territory"),
    "Vendor.csv":              ("Purchasing", "Vendor",              "dim_vendor"),
}

In [0]:
import re
from pyspark.sql.types import (
    StructField, StructType, IntegerType, DoubleType, StringType, TimestampType
)

with open(ddl_file, "r", encoding="utf-8") as f:
    ddl_text = f.read()

# SQL Server type -> Spark type. Matches the typing already used in bronze
# (money/decimal as DOUBLE, flag-like columns as INT, dates as TIMESTAMP).
sql_type_map = {
    "int": IntegerType(), "smallint": IntegerType(), "tinyint": IntegerType(), "flag": IntegerType(),
    "money": DoubleType(), "float": DoubleType(), "real": DoubleType(),
    "decimal": DoubleType(), "numeric": DoubleType(),
    "name": StringType(), "accountnumber": StringType(), "phone": StringType(), "ordernumber": StringType(),
    "nvarchar": StringType(), "nchar": StringType(), "varchar": StringType(), "char": StringType(),
    "uniqueidentifier": StringType(), "xml": StringType(), "sysname": StringType(),
    "datetime": TimestampType(), "smalldatetime": TimestampType(), "datetime2": TimestampType(),
    "date": TimestampType(),
}

# Column line: [ColName] [type] ... or [ColName] bareword ... (uniqueidentifier)
column_pattern = re.compile(r"^\[([A-Za-z0-9_]+)\]\s+(?:\[([A-Za-z0-9_]+)\]|([A-Za-z]+))")

def parse_table_schema(ddl_schema, ddl_table):
    """Extract column names (in DDL order) and Spark types from the CREATE TABLE block."""
    match = re.search(
        r"CREATE TABLE \[" + ddl_schema + r"\]\.\[" + ddl_table + r"\]\((.*?)\n\)",
        ddl_text, re.DOTALL,
    )
    if not match:
        raise ValueError(f"CREATE TABLE block for [{ddl_schema}].[{ddl_table}] not found in {ddl_file}")

    fields = []
    for line in match.group(1).split("\n"):
        line = line.strip()
        if not line.startswith("["):
            continue  # skip CONSTRAINT ... lines
        col_match = column_pattern.match(line)
        if not col_match:
            continue
        column_name = col_match.group(1)
        type_token = (col_match.group(2) or col_match.group(3) or "").lower()
        if type_token == "as":
            # Computed column (e.g. [LineTotal] AS ...) — money/decimal arithmetic in this database
            spark_type = DoubleType()
        else:
            spark_type = sql_type_map.get(type_token)
            if spark_type is None:
                raise ValueError(f"Unmapped SQL type '{type_token}' for {ddl_schema}.{ddl_table}.{column_name}")
        fields.append(StructField(column_name, spark_type, True))
    return StructType(fields)

# Preview the column names extracted from the DDL for every table
for csv_name, (ddl_schema, ddl_table, bronze_table) in table_configs.items():
    schema = parse_table_schema(ddl_schema, ddl_table)
    print(f"{bronze_table} ({len(schema.fields)} columns): " + ", ".join(f.name for f in schema.fields))

dim_product (25 columns): ProductID, Name, ProductNumber, MakeFlag, FinishedGoodsFlag, Color, SafetyStockLevel, ReorderPoint, StandardCost, ListPrice, Size, SizeUnitMeasureCode, WeightUnitMeasureCode, Weight, DaysToManufacture, ProductLine, Class, Style, ProductSubcategoryID, ProductModelID, SellStartDate, SellEndDate, DiscontinuedDate, rowguid, ModifiedDate
dim_purchaseorderdetail (11 columns): PurchaseOrderID, PurchaseOrderDetailID, DueDate, OrderQty, ProductID, UnitPrice, LineTotal, ReceivedQty, RejectedQty, StockedQty, ModifiedDate
dim_purchaseorderheader (13 columns): PurchaseOrderID, RevisionNumber, Status, EmployeeID, VendorID, ShipMethodID, OrderDate, ShipDate, SubTotal, TaxAmt, Freight, TotalDue, ModifiedDate
fact_sales (11 columns): SalesOrderID, SalesOrderDetailID, CarrierTrackingNumber, OrderQty, ProductID, SpecialOfferID, UnitPrice, UnitPriceDiscount, LineTotal, rowguid, ModifiedDate
dim_sales_territory (10 columns): TerritoryID, Name, CountryRegionCode, Group, SalesYTD, S

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

for csv_name, (ddl_schema, ddl_table, bronze_table) in table_configs.items():
    source_path = f"{source_volume}/{csv_name}"
    target_table = f"{target_schema}.{bronze_table}"

    # Column names/types come from the .sql DDL; raw files are TSVs without a header row
    df = (
        spark.read
        .option("header", False)
        .option("sep", "\t")
        .schema(parse_table_schema(ddl_schema, ddl_table))
        .csv(source_path)
    )

    rows_written = df.count()

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(target_table))

    print(f"{bronze_table}: {rows_written:,} rows written to {target_table}")

dim_product: 504 rows written to adventure_works.bronze.dim_product
dim_purchaseorderdetail: 8,845 rows written to adventure_works.bronze.dim_purchaseorderdetail
dim_purchaseorderheader: 4,012 rows written to adventure_works.bronze.dim_purchaseorderheader
fact_sales: 121,317 rows written to adventure_works.bronze.fact_sales
dim_sales_territory: 10 rows written to adventure_works.bronze.dim_sales_territory
dim_vendor: 104 rows written to adventure_works.bronze.dim_vendor


In [0]:
verification_rows = []
for csv_name, (ddl_schema, ddl_table, bronze_table) in table_configs.items():
    ddl_columns = [f.name for f in parse_table_schema(ddl_schema, ddl_table).fields]
    bronze_df = spark.table(f"{target_schema}.{bronze_table}")
    verification_rows.append((
        bronze_table,
        len(ddl_columns),
        bronze_df.count(),
        bronze_df.columns == ddl_columns,
    ))

verification_df = spark.createDataFrame(
    verification_rows,
    schema="table string, ddl_columns int, bronze_rows long, columns_match boolean",
)
display(verification_df)

mismatched = verification_df.filter("columns_match = false")
assert mismatched.count() == 0, f"DDL/bronze column mismatches: {mismatched.collect()}"
print("Success: all 6 bronze tables loaded with column names from the DDL.")

table,ddl_columns,bronze_rows,columns_match
dim_product,25,504,true
dim_purchaseorderdetail,11,8845,true
dim_purchaseorderheader,13,4012,true
fact_sales,11,121317,true
dim_sales_territory,10,10,true
dim_vendor,8,104,true


Success: all 6 bronze tables loaded with column names from the DDL.
